[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eliem08/ai-agents-healthcare-labs/blob/main/notebooks/Lab08_Capstone.ipynb)

# Lab 8 — Capstone: design and pitch your laboratory agent
**AI Agents for Healthcare and Diagnostics** · Module 7 · about 2 h 15 · teams of 3–4

**The situation.** You now know the patterns: tools that compute, status rules, guardrails in code, human
checkpoints, a guardian, audit logs, measurement against an answer key. Choose a **real task from your own work**,
design an agent for it, prototype it (with or without code), and pitch it in 5 minutes.

**Plan:** choose a task (10 min) → design canvas (45 min) → prototype: Track A in Google AI Studio Build or
Track B in this notebook (45 min) → 5-minute pitches + scoring (35 min).

**Example tasks:** batch data review · instrument troubleshooting assistant · assay development documentation ·
method transfer · literature monitoring · specimen reception · result interpretation comments (with a guardian).

**Success criteria (from the course outline):** every design has at least one **human checkpoint** and one
**guardian check**, names its **compliance controls** and its **success metrics**.

### Setup — run this cell (about 1 minute)
Same as in Lab 0: it installs the course tools, downloads the course files and checks your key. The code is folded: just run it.

In [ ]:
# @title Setup — click run and wait about 1 minute
import os, subprocess, sys
REPO = "https://github.com/eliem08/ai-agents-healthcare-labs"
MODEL = "gemini-3.5-flash-lite"  # @param {type:"string"}
___ = "___"  # the blank you will replace in YOUR CELL

def _step(what, cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"FAILED: {what}. Run this cell again; if it fails twice, send the text below to the trainer:")
        print((r.stderr or r.stdout)[-800:])
        return False
    print(f"done: {what}")
    return True

try:
    import google.colab  # noqa: F401  — present only inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

_ready = True
if IN_COLAB:
    if not os.path.exists("/content/course"):
        _ready = _step("download course files", ["git", "clone", "-q", "--depth", "1", REPO, "/content/course"])
    if _ready:
        _ready = _step("install Google ADK", [sys.executable, "-m", "pip", "install", "-q", "google-adk==2.15.0"])
    sys.path.insert(0, "/content/course")

SETUP_OK, SETUP_RESULTS = False, []
if _ready:
    from google.adk.agents import Agent
    from labkit.setup import check_environment, show_checks
    from labkit.agents import make_model, run, DETERMINISTIC
    from labkit.ui import check, has_blank, ready, copy_box, done_card
    import os, re, tempfile
    from IPython.display import HTML, display
    from labkit import capstone
    from labkit.audit import AuditLog
    from labkit.ui import error_line
    LOG = AuditLog()
    SETUP_RESULTS = check_environment(MODEL, ping_pubmed=False)
    SETUP_OK = show_checks(SETUP_RESULTS)

**What you should see:** every line marked **OK**, then *"Everything is ready"*. Same Setup as Lab 0.

**If it does not work**

| You see | Do this |
|---|---|
| FIX — API key found | Open Secrets (key icon, left sidebar): `GOOGLE_API_KEY`, *Notebook access* on; run Setup again |
| FIX — API key works, "busy" or "quota" | Wait 30–60 seconds, run Setup again |
| FAILED: download / install | Run Setup again; if it fails twice, send the text to the trainer |
| `NameError: ... is not defined` in later cells | Session restarted: click the cell, then **Runtime → Run before** |

### Task 1 — Fill in the agent design canvas
<sub>IN THE NOTEBOOK · ABOUT 45 MIN</sub>

**Your task.** Design before you build: one page that a supervisor, a quality manager and an IT colleague can all read.

**Steps**

1. Type your team name and fill the 11 boxes in the form (one or two sentences each).
2. Run the cell and the CHECK: it applies the success criteria and shows your one-page canvas.
3. Download the canvas (Markdown) to paste into Google Docs or Slides for the pitch.
4. **Teams:** one scribe fills the canvas and shares it as a Google Doc; the others paste the same text into their own form, so everyone's CHECK and status card work.

In [ ]:
# @title Your agent design canvas
# YOUR CELL — fill in the form on the right, then run this cell
team_name = ""  # @param {type:"string"}
problem_and_users = ""  # @param {type:"string"}
goal = ""  # @param {type:"string"}
inputs_and_data = ""  # @param {type:"string"}
tools = ""  # @param {type:"string"}
agent_steps = ""  # @param {type:"string"}
human_checkpoints = ""  # @param {type:"string"}
guardian_checks = ""  # @param {type:"string"}
risks = ""  # @param {type:"string"}
compliance_controls = ""  # @param {type:"string"}
success_metrics = ""  # @param {type:"string"}
pilot_plan = ""  # @param {type:"string"}
test_cases = ""  # @param {type:"string"}

In [ ]:
# @title CHECK — run this cell: it tells you what is right and what to fix
CANVAS = {k: globals()[k] for k in ['problem_and_users', 'goal', 'inputs_and_data', 'tools', 'agent_steps', 'human_checkpoints', 'guardian_checks', 'risks', 'compliance_controls', 'success_metrics', 'pilot_plan', 'test_cases']}
_ok = check(capstone.check_canvas(CANVAS) + [(bool(team_name.strip()), "Team name", "Type your team name.")])
display(HTML(capstone.canvas_html(team_name or "your team", CANVAS)))
if _ok:
    CANVAS_OK = True
    _p = os.path.join("/tmp" if IN_COLAB else tempfile.gettempdir(), "agent_canvas.md")
    _md = capstone.canvas_markdown(team_name, CANVAS)
    if _md != globals().get("_LAST_CANVAS_MD"):
        open(_p, "w", encoding="utf-8").write(_md)
        globals()["_LAST_CANVAS_MD"] = _md
        if IN_COLAB:
            from google.colab import files
            files.download(_p)
        print("Canvas saved:", _p)
    else:
        print("Canvas unchanged — not downloaded again.")

<details><summary><b>Hint</b> — open only if you are stuck</summary>

What each box needs:

- **Problem and users** — Who has which problem today?
- **Goal** — What will be better, by how much?
- **Inputs and data** — Which data does the agent read? Where from?
- **Tools** — Which tools (functions) does it call?
- **Agent steps** — What does it do, step by step?
- **Human checkpoints** — Where does a person approve or decide?
- **Guardian checks** — Which checks run on its output?
- **Risks** — What can go wrong?
- **Compliance controls** — ISO 15189 / CAP / CLIA / GDPR-HIPAA / IVDR / audit trail / validation
- **Success metrics** — Measurable targets (with numbers)
- **Pilot plan** — How will you try it safely?
- **Test cases** — 3 inputs and the status you expect for each (your first answer key)

Reuse the course patterns: tools compute (Lab 2), guardrails in code (Lab 3), thresholds and a review queue (Lab 4), validated data exchange (Lab 5), a guardian (Lab 6), verified citations (Lab 7).

</details>

## Prototype (45 minutes)
**Track A — no code:** a clickable prototype with Google AI Studio *Build* (Task 2).
**Track B — notebook:** a minimal working agent with your own tool, instruction and test cases (Task 3).

In [ ]:
# @title Track A: a prompt for AI Studio Build, generated from your canvas — run this cell
if "CANVAS" not in globals():
    display(error_line("Fill and check the canvas first (Task 1)."))
else:
    _prompt = (f"Build a simple web app prototype for a laboratory AI assistant. Problem: {CANVAS['problem_and_users']} "
               f"Goal: {CANVAS['goal']} Inputs: {CANVAS['inputs_and_data']} Steps: {CANVAS['agent_steps']} "
               f"Show clearly where a human approves: {CANVAS['human_checkpoints']} "
               f"Before showing a result, run these checks and show their outcome: {CANVAS['guardian_checks']} "
               f"Include a 'Test cases' screen that runs these examples and shows expected vs actual status: "
               f"{CANVAS['test_cases']} Display a status for each item and "
               "never present the result as final without human approval. Use only made-up example data.")
    display(copy_box(_prompt, "Paste into aistudio.google.com → Build"))

### Task 2 — Track A — prototype with AI Studio Build
<sub>IN THE GEMINI APP · ABOUT 45 MIN</sub>

**Your task.** Turn the canvas into something people can click, without writing code.

**Steps**

1. Open [aistudio.google.com](https://aistudio.google.com), choose **Build**, paste the prompt above.
2. Iterate in the chat until the human checkpoint is visible on screen. Use made-up data only.
3. Optional for the pitch: ask **Nano Banana** for a one-image workflow visual; ask **Gemini Notebook** for an Audio Overview of your canvas — if the summary misses your human checkpoint, so will your audience.
4. Paste your app's share link (or 'not available'), run the cell and the CHECK.

In [ ]:
# @title Your prototype link
# YOUR CELL — fill in the form on the right, then run this cell
prototype_link = ""  # @param {type:"string"}

In [ ]:
# @title CHECK — run this cell: it tells you what is right and what to fix
_ok = check([(len(prototype_link.strip()) >= 3, "Link or 'not available' entered", "Paste the share link.")])
if _ok:
    PROTOTYPE_LINK = prototype_link.strip()

<details><summary><b>Hint</b> — open only if you are stuck</summary>

AI Studio apps keep your API key server-side and can be shared with colleagues; the app's calls count towards your quota. Keep it a prototype: synthetic data, no real patients.

</details>

### Task 3 — Track B — a minimal working agent
<sub>IN THE NOTEBOOK · ABOUT 45 MIN</sub>

**Your task.** Prove the core of your design works: one tool, status rules, test cases with expected results.

**Steps**

1. Start from the worked example (a specimen-reception checker built on SOP-02) or replace it with your own: a few example records, one tool that computes, an instruction with status rules.
2. Write at least 3 **test cases with the expected status** before running — like the answer keys of this course.
3. Run your cell and the CHECK; the next cell builds the agent, runs your test cases and shows the audit log.

In [ ]:
# YOUR CELL — replace every ___ , then run this cell
# Replace the example with your task. Keep: a tool that computes, status rules, TEST_CASES with expected status.
SAMPLES = {"S1": {"analyte": "potassium", "h_index": 40, "hours_since_collection": 2, "stored": "room temperature", "volume_ml": 1.0},
           "S2": {"analyte": "potassium", "h_index": 180, "hours_since_collection": 1, "stored": "room temperature", "volume_ml": 1.0}}

def check_sample(sample_id: str) -> dict:
    """Apply the SOP-02 acceptance rules to one received specimen and return the problems found."""
    s = SAMPLES.get(sample_id)
    if s is None:
        return {"status": "error", "message": f"Unknown sample {sample_id}"}
    problems = []
    if s["analyte"] == "potassium" and s["h_index"] > 100:
        problems.append("H-index above 100: potassium rejected (SOP-02 §3)")
    return {"sample_id": sample_id, "problems": problems}

PROTOTYPE_INSTRUCTION = """___"""
TEST_CASES = [{"input": "Check sample S1.", "expected": "___"},
              {"input": "Check sample S2.", "expected": "___"}]
TOOLS = [check_sample]

In [ ]:
# @title CHECK — run this cell: it tells you what is right and what to fix
_blank = has_blank(PROTOTYPE_INSTRUCTION) or any(has_blank(c.get("expected", "")) for c in TEST_CASES)
_ok = check([(not _blank, "Instruction and expected statuses written", _blank or ""),
             (isinstance(TOOLS, list) and all(callable(t) for t in TOOLS) and TOOLS, "Tools are functions",
              "TOOLS = [your_function] (no quotes, no ())."),
             ("status:" in PROTOTYPE_INSTRUCTION.lower(), "Fixed STATUS line",
              "End the instruction with: STATUS: <your statuses>."),
             (len(TEST_CASES) >= 3, f"At least 3 test cases ({len(TEST_CASES)})", "Add test cases with expected status.")])

<details><summary><b>Hint</b> — open only if you are stuck</summary>

Smaller is better: one tool that really computes beats five that pretend. Your test cases are your first answer key — include at least one case that must be refused or escalated.

</details>

In [ ]:
# @title Build your agent and run your test cases — run this cell
if has_blank(PROTOTYPE_INSTRUCTION) or not ready("prototype", PROTOTYPE_INSTRUCTION, TOOLS, example="check_sample"):
    display(error_line("Finish Task 3 first."))
else:
    prototype = Agent(name="prototype", model=make_model(MODEL), instruction=PROTOTYPE_INSTRUCTION, tools=TOOLS,
                      generate_content_config=DETERMINISTIC, after_tool_callback=LOG.callback)
    PROTO_RESULTS = capstone.run_test_cases(prototype, TEST_CASES)
    display(HTML(capstone.results_html(PROTO_RESULTS)))
    display(HTML(LOG.html()))

## Pitch (5 minutes per team) and peer scoring
**Structure:** 1) the problem and who has it · 2) the canvas or the prototype, live · 3) where the human decides
and what the guardian checks · 4) risks and compliance controls · 5) success metrics and pilot plan.

**Scoring grid (1–5 each):** value for the laboratory · feasibility · safety and compliance · clarity of human
oversight.

| Score | Value | Feasibility | Safety and compliance | Human oversight |
|---|---|---|---|---|
| **1** | nobody's real problem | needs data or systems that do not exist | no guardian, no controls named | unclear who decides |
| **3** | a real problem, benefit not measured | possible with work; data access unclear | some checks and frameworks named | a checkpoint exists, role or action vague |
| **5** | real problem, measurable target | could pilot next month with existing data | guardian checks + validation, audit trail, data protection | named role approves named outputs before release |

### Task 4 — Score the other teams
<sub>IN THE NOTEBOOK · ABOUT 5 MIN</sub>

**Your task.** Give structured feedback with the same grid for every team.

**Steps**

1. After each pitch, add a line: `Team name: value, feasibility, safety, oversight` (scores 1–5).
2. Run your cell and the CHECK; post the table in the course chat at the end.

In [ ]:
# @title Your scores (one team per line: name: value, feasibility, safety, oversight)
# YOUR CELL — replace every ___ , then run this cell
team_scores = """___"""

In [ ]:
# @title CHECK — run this cell: it tells you what is right and what to fix
_scores = {}
for _line in team_scores.splitlines():
    _m = re.match(r"\s*(.+?):\s*([1-5])\s*,\s*([1-5])\s*,\s*([1-5])\s*,\s*([1-5])\s*$", _line)
    if _m:
        _scores[_m.group(1)] = dict(zip(("value", "feasibility", "safety", "oversight"), map(int, _m.groups()[1:])))
_ok = check([(bool(_scores), f"{len(_scores)} team(s) scored", "Write lines like: Team QC: 4, 3, 5, 4")])
if _ok:
    SCORES_DONE = True
    display(HTML(capstone.scores_html(_scores)))

<details><summary><b>Hint</b> — open only if you are stuck</summary>

Score oversight strictly: can you point to the exact moment a person approves, and what they see then?

</details>

## Closing the course
1. **Tools compute, agents decide what to call, humans decide what is released.**
2. **Guardrails in code, guardians on outputs, audit logs by callbacks** — and every claim traceable to a source.
3. **Measure before you trust:** answer keys, detection rates, false alarms, missed high-severity cases.
4. **Where to go next:** Google ADK documentation; Gemini Enterprise Agent Designer (if your organisation enables
   it); Google Health AI models (MedGemma, MedSigLIP — need GPUs and their own validation); your quality system.

## Finish — your status card
Type your name, run the cell, post a screenshot in the course chat.

In [ ]:
# @title Status card — type your name, run, then screenshot the card
participant_name = ""  # @param {type:"string"}
import datetime
_g = globals()
_pr = _g.get("PROTO_RESULTS")
_lines = [f"Participant: {participant_name.strip() or 'NAME MISSING — type it in the box'}",
          f"Date: {datetime.date.today().isoformat()} · team {_g.get('team_name', '?')}",
          f"{'OK ' if 'CANVAS_OK' in _g else 'FIX'} — Design canvas meets the success criteria (Task 1)",
          f"{'OK ' if 'PROTOTYPE_LINK' in _g else 'n/a'} — AI Studio prototype (Track A)",
          f"{'OK ' if _pr else 'n/a'} — Notebook prototype: "
          + (f"{sum(r['ok'] for r in _pr)}/{len(_pr)} test cases right" if _pr else "(Track B)"),
          f"{'OK ' if 'SCORES_DONE' in _g else 'FIX'} — Peer scoring (Task 4)"]
done_card("Lab 8 — status", _lines, complete=bool(participant_name.strip()) and "CANVAS_OK" in _g
          and ("PROTOTYPE_LINK" in _g or bool(_pr)) and "SCORES_DONE" in _g, complete_text="Complete — course finished")

*Course material for training only. Use synthetic data in prototypes; real patient data requires your
organisation's approvals.*